# Predict a finished batch's quality from the historical boxes

**Not a trained model - a lookup.** `search()` builds boxes (X range + mean Y + Cp).
`predict.score(res, x)` finds which boxes a finished batch falls into and returns the
expected Viscosity / Kappa. See `CONCEPT.md` for the idea and the backtest result.

Predict **after discharge** (all X final); the lab Y is still hours away, so it is a
genuine look-ahead. No PI here - X comes from you (dict / csv).

In [1]:
import importlib
import pandas as pd

import config, golden, capability as cap, predict
for mod in (config, golden, cap, predict):
    importlib.reload(mod)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

df = golden.load_golden()

golden data: 760 batches with Kappa + Viscosity (2026-05-09 .. 2026-10-18), golden 309 (41%)


## 1. One batch -> which boxes, which Y?
Edit `x` to the finished batch's X. Keys can be column names or short labels.

In [2]:
x = {
    "pfactor_overshoot": 40,
    "alkali_accumulator1_conc_mean_hot_liquor_filling": 95,
    "prehydrolysis_minutes": 70,
    "chip_weight_max_chip_filling": 36,
    "digester": 3,
    "pfactor_temp_setpoint_last_prehydrolysis": 165,
}

scored, cons = predict.predict(x)          # build res (same-digester) + score
display(scored)
print("consensus:", cons)

golden data: 760 batches with Kappa + Viscosity (2026-05-09 .. 2026-10-18), golden 309 (41%)
search: 71 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)


,combo,rules,n,visc_pred,kappa_pred,cp_visc,cp_kappa,cp_min
0,Prehyd minutes x P-temp SP,"Prehyd minutes 63.229-75.427, P-temp SP 164-166",52,759.2,9.79,0.72,0.67,0.67


consensus: {'visc_pred': 759.2, 'kappa_pred': 9.79, 'cp_min': 0.67, 'n_boxes': 1, 'n_batches': 52}


## 2. All boxes it enters (digester as an X too)
Score against every box, not just same-digester - shows all windows the batch sits in.

In [3]:
res_all = cap.search(df)
scored_all = predict.score(res_all, x)
display(scored_all)
print("consensus:", predict.consensus(scored_all))

search: 137 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)


,combo,rules,n,visc_pred,kappa_pred,cp_visc,cp_kappa,cp_min
0,Digester x P-temp SP,"Digester 3, P-temp SP 164-166",55,757.8,9.79,0.72,0.56,0.56
1,Prehyd minutes x Digester,"Prehyd minutes 56.081-75.067, Digester 3",73,759.7,9.88,0.62,0.56,0.56
2,Prehyd minutes x P-temp SP,"Prehyd minutes 56.081-75.067, P-temp SP 164-166",103,757.1,9.86,0.58,0.47,0.47
3,P-factor overshoot x Prehyd minutes,"P-factor overshoot 25.915-62.096, Prehyd minutes 56.081-...",32,771.3,10.28,0.38,0.42,0.38
4,P-factor overshoot x Digester,"P-factor overshoot 25.915-62.096, Digester 3",69,807.1,10.45,0.38,0.35,0.35


consensus: {'visc_pred': 769.5, 'kappa_pred': 10.02, 'cp_min': 0.35, 'n_boxes': 5, 'n_batches': 332}


## 3. Backtest - is the lookup better than guessing the mean?
Leave-one-out over golden. Beats the baseline MAE = the method carries real signal.
(Slow: rebuilds res per batch.)

In [4]:
predict.backtest(by_digester=True)

golden data: 760 batches with Kappa + Viscosity (2026-05-09 .. 2026-10-18), golden 309 (41%)
search: 71 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 71 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 28 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 71 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 71 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 28 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 71 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 28 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 28 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 71 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 71 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 28 boxes with N >= 15  (combo size 2, 3 bins/X, ranked by cp_min)
search: 51 boxes wi

{'covered': 760,
 'n': 760,
 'mae_visc': np.float64(27.667763157894736),
 'base_visc': np.float64(33.03249307479224),
 'mae_kappa': np.float64(0.676907894736842),
 'base_kappa': np.float64(0.7121342451523546),
 'spec_agreement': np.float64(0.7605263157894737)}